In [1]:
%pip install chromadb langchain langchain-community langchain-groq sentence-transformers ragas datasets groq -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, SystemMessage
from sentence_transformers import SentenceTransformer
from IPython.display import display, Markdown

In [7]:
import os
from dotenv import load_dotenv

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("No se encontró GROQ_API_KEY en el archivo .env")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

In [8]:
df = pd.read_csv("../data/processed/df_master_final.csv")
print(df.shape)
df.head()

(112, 14)


,comuna,anio,total_delitos,victimas_viales,muertos_viales,extorsion,homicidio,hurto_a_persona,hurto_a_residencia,hurto_de_carro,hurto_de_moto,hurto_de_semoviente,bienestar_score,nivel_bienestar
0,1.0,2015,243,583,5,6,8,95,52,6,76,0,0.810070,alto
1,1.0,2016,282,591,3,8,15,159,33,6,61,0,0.857138,alto
2,1.0,2017,358,463,4,6,12,167,103,4,66,0,0.860027,alto
3,1.0,2018,457,517,5,24,12,216,105,6,94,0,0.803345,alto
4,1.0,2019,541,591,4,14,21,303,82,11,110,0,0.796878,alto


In [18]:
documentos = []
ids = []

for _, row in df.iterrows():
    texto = (
        f"En el año {int(row['anio'])}, la comuna {int(row['comuna'])} de Medellín "
        f"registró {int(row['total_delitos'])} delitos totales, "
        f"{int(row['victimas_viales'])} víctimas y {int(row['muertos_viales'])} muertos en incidentes viales. "
        f"Delitos específicos: {int(row['homicidio'])} homicidios, "
        f"{int(row['hurto_a_persona'])} hurtos a persona, "
        f"{int(row['extorsion'])} extorsiones, "
        f"{int(row['hurto_a_residencia'])} hurtos a residencia, "
        f"{int(row['hurto_de_carro'])} hurtos de carro, "
        f"{int(row['hurto_de_moto'])} hurtos de moto. "
        f"El índice de bienestar fue de {row['bienestar_score']:.4f}, "
        f"correspondiente al nivel '{row['nivel_bienestar']}'."
    )
    documentos.append(texto)
    ids.append(f"doc_{int(row['anio'])}_{int(row['comuna'])}")

print(f"Total de documentos generados: {len(documentos)}")
print("\nEjemplo:")
print(documentos[0])

Total de documentos generados: 112

Ejemplo:
En el año 2015, la comuna 1 de Medellín registró 243 delitos totales, 583 víctimas y 5 muertos en incidentes viales. Delitos específicos: 8 homicidios, 95 hurtos a persona, 6 extorsiones, 52 hurtos a residencia, 6 hurtos de carro, 76 hurtos de moto. El índice de bienestar fue de 0.8101, correspondiente al nivel 'alto'.


In [20]:
# Modelo de embeddings
modelo_embeddings = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="BAAI/bge-m3"
)

# Cliente ChromaDB en memoria
cliente_chroma = chromadb.Client()

# Crear colección
colecciones_existentes = [c.name for c in cliente_chroma.list_collections()]

if "medellinvive" in colecciones_existentes:
    cliente_chroma.delete_collection(name="medellinvive")
    print("Colección anterior eliminada.")

coleccion = cliente_chroma.create_collection(
    name="medellinvive",
    embedding_function=modelo_embeddings
)

BATCH_SIZE = 50
for i in range(0, len(documentos), BATCH_SIZE):
    coleccion.add(
        documents=documentos[i:i+BATCH_SIZE],
        ids=ids[i:i+BATCH_SIZE]
    )

print(f"Documentos indexados en ChromaDB: {coleccion.count()}")

Colección anterior eliminada.
Documentos indexados en ChromaDB: 112


In [21]:
def recuperar_contexto(pregunta: str, n_resultados: int = 5) -> str:
    """Busca los documentos más relevantes en ChromaDB para una pregunta."""
    resultados = coleccion.query(
        query_texts=[pregunta],
        n_results=n_resultados
    )
    fragmentos = resultados["documents"][0]
    contexto = "\n".join([f"- {f}" for f in fragmentos])
    return contexto

In [22]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2,
    max_tokens=512,
    api_key=GROQ_API_KEY
)

In [24]:
def responder_pregunta(pregunta: str) -> str:
    """Pipeline RAG: recupera contexto y genera respuesta con LLaMA 3.1."""
    
    # 1. Retrieval
    contexto = recuperar_contexto(pregunta)
    
    # 2. Prompt
    system_prompt = (
        "Eres un asistente experto en análisis urbano de Medellín, Colombia. "
        "Responde preguntas ciudadanas usando ÚNICAMENTE la información del contexto proporcionado. "
        "Si la información no está en el contexto, dilo claramente. "
        "Responde siempre en español, de forma clara y concisa."
    )
    
    user_prompt = f"""Contexto con datos de Medellín:
{contexto}

Pregunta del ciudadano: {pregunta}

Responde basándote solo en el contexto anterior."""
    
    # 3. Generación
    mensajes = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=user_prompt)
    ]
    respuesta = llm.invoke(mensajes)
    
    return respuesta.content

In [ ]:
preguntas_demo = [
    "¿Cuál fue la comuna con más delitos en 2020?",
    "¿Cómo evolucionó la accidentalidad vial en la comuna 10 entre 2015 y 2021?",
    "¿Qué comunas tuvieron nivel de bienestar BAJO en 2019?",
    "¿En qué año tuvo la comuna 13 su mejor índice de bienestar?",
]

for pregunta in preguntas_demo:
    print(f"\n{'='*60}")
    print(f"🔍 Pregunta: {pregunta}")
    print(f"{'='*60}")
    respuesta = responder_pregunta(pregunta)
    print(f"🤖 Respuesta:\n{respuesta}")


🔍 Pregunta: ¿Cuál fue la comuna con más delitos en 2020?
🤖 Respuesta:
Según el contexto proporcionado, la comuna con más delitos en 2020 fue la comuna 10 de Medellín, con un total de 9364 delitos.

🔍 Pregunta: ¿Cómo evolucionó la accidentalidad vial en la comuna 10 entre 2015 y 2021?
🤖 Respuesta:
Según el contexto proporcionado, en la comuna 10 de Medellín:

- En el año 2015, se registraron 63 muertos en incidentes viales.
- En el año 2021, se registraron 41 muertos en incidentes viales.

La accidentalidad vial en la comuna 10 disminuyó entre 2015 y 2021, ya que el número de muertos en incidentes viales pasó de 63 a 41.

🔍 Pregunta: ¿Qué comunas tuvieron nivel de bienestar BAJO en 2019?
🤖 Respuesta:
Según el contexto proporcionado, las comunas que tuvieron nivel de bienestar BAJO en 2019 fueron:

- Comuna 4: con un índice de bienestar de -0.4419.
- Comuna 5: con un índice de bienestar de -0.9348.
- Comuna 7: con un índice de bienestar de -0.1699.
- Comuna 11: con un índice de bienesta

In [32]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_groq import ChatGroq
from langchain_community.embeddings import HuggingFaceEmbeddings
from datasets import Dataset

ragas_llm = LangchainLLMWrapper(ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2,
    api_key=GROQ_API_KEY
))

ragas_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
)

# Dataset de evaluación
eval_data = {
    "question": [
        "¿Cuál fue la comuna con más delitos en 2018?",
        "¿Qué nivel de bienestar tuvo la comuna 1 en 2021?",
        "¿Cuántas víctimas viales hubo en la comuna 7 en 2017?",
    ],
    "answer": [],
    "contexts": [],
    "ground_truth": [
        "La comuna con más delitos en 2018 fue la que presentó el mayor total_delitos en ese año.",
        "El nivel de bienestar de la comuna 1 en 2021 fue alto.",
        "El dato de víctimas viales de la comuna 7 en 2017 está disponible en el dataset maestro.",
    ]
}

for pregunta in eval_data["question"]:
    contexto = recuperar_contexto(pregunta)
    respuesta = responder_pregunta(pregunta)
    eval_data["contexts"].append([contexto])
    eval_data["answer"].append(respuesta)

dataset_eval = Dataset.from_dict(eval_data)

# Evaluar con Groq en lugar de OpenAI
resultados_ragas = evaluate(
    dataset_eval,
    metrics=[faithfulness, answer_relevancy],
    llm=ragas_llm,
    embeddings=ragas_embeddings
)

print("\n📊 Resultados RAGAS:")
print(resultados_ragas)

C:\Users\USUARIO\AppData\Local\Temp\ipykernel_1276\554959858.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_1276\554959858.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_1276\554959858.py:9: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'


📊 Resultados RAGAS:
{'faithfulness': 0.7222, 'answer_relevancy': 0.8187}


In [33]:
for i, (p, r) in enumerate(zip(eval_data["question"], eval_data["answer"])):
    print(f"Q{i+1}: {p}")
    print(f"A{i+1}: {r}\n")

Q1: ¿Cuál fue la comuna con más delitos en 2018?
A1: Según el contexto proporcionado, la comuna con más delitos en 2018 fue la comuna 10 de Medellín, con un total de 13,495 delitos.

Q2: ¿Qué nivel de bienestar tuvo la comuna 1 en 2021?
A2: Según el contexto proporcionado, el índice de bienestar de la comuna 1 de Medellín en el año 2021 fue de 0.8661, correspondiente al nivel 'alto'.

Q3: ¿Cuántas víctimas viales hubo en la comuna 7 en 2017?
A3: Según el contexto proporcionado, en el año 2017, la comuna 7 de Medellín registró 2226 víctimas en incidentes viales.



In [34]:
display(Markdown(f"""
## ✅ Resumen del Sistema RAG — MedellínVive

| Componente | Detalle |
|---|---|
| **Documentos indexados** | {coleccion.count()} |
| **Modelo de embeddings** | BAAI/bge-m3 |
| **Vector store** | ChromaDB (en memoria) |
| **LLM** | LLaMA 3.1 8B vía Groq |
| **Métricas RAGAS** | Faithfulness + Answer Relevancy |

\n

El sistema es capaz de responder preguntas ciudadanas sobre bienestar urbano
en Medellín usando datos reales de criminalidad y accidentalidad vial (2015-2021).
"""))


## ✅ Resumen del Sistema RAG — MedellínVive

| Componente | Detalle |
|---|---|
| **Documentos indexados** | 112 |
| **Modelo de embeddings** | BAAI/bge-m3 |
| **Vector store** | ChromaDB (en memoria) |
| **LLM** | LLaMA 3.1 8B vía Groq |
| **Métricas RAGAS** | Faithfulness + Answer Relevancy |




El sistema es capaz de responder preguntas ciudadanas sobre bienestar urbano
en Medellín usando datos reales de criminalidad y accidentalidad vial (2015-2021).


In [ ]:
historial = []

def conversar(pregunta: str) -> str:
    contexto = recuperar_contexto(pregunta)
    
    mensajes = [
        SystemMessage(content=(
            "Eres un asistente experto en análisis urbano de Medellín, Colombia. "
            "Responde preguntas ciudadanas usando ÚNICAMENTE la información del contexto proporcionado. "
            "Si la información no está en el contexto, dilo claramente. "
            "Responde siempre en español, de forma clara y concisa."
        ))
    ]
    
    for turno in historial:
        mensajes.append(HumanMessage(content=turno["pregunta"]))
        mensajes.append(SystemMessage(content=turno["respuesta"]))
    
    mensajes.append(HumanMessage(content=(
        f"Contexto con datos de Medellín:\n{contexto}\n\n"
        f"Pregunta: {pregunta}"
    )))
    
    respuesta = llm.invoke(mensajes).content
    historial.append({"pregunta": pregunta, "respuesta": respuesta})
    
    return respuesta


print("🤖 MedellínVive — Asistente Urbano")
print("Escribe 'salir' para terminar\n")

while True:
    pregunta = input("Tú: ")
    
    if not pregunta:
        continue
    elif pregunta.lower() == "salir":
        print("Hasta luego.")
        break
    elif pregunta.lower() == "limpiar":
        historial.clear()
        print("🧹 Historial limpiado.\n")
        continue
    
    respuesta = conversar(pregunta)
    print(f"\n🤖 Asistente: {respuesta}\n")

🤖 MedellínVive — Asistente Urbano
Escribe 'salir' para terminar

